# Notebook 04 — DeepFM vs FM Comparison

Extends the FM baseline with a deep MLP tower sharing the **same embedding table**.  
Run **after** `03_train_fm_model.ipynb` so the FM baseline RMSE is already available.

**Output:** `models/deepfm_model.pt`, `models/deepfm_config.json`, `models/item_embeddings.npy` (overwritten with DeepFM vectors if DeepFM wins), `models/comparison.csv`

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from src.fm_model import FactorizationMachine, DeepFM
from src.data_utils import load_ratings, load_movies, build_id_mappings, train_test_split_by_time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Load Data (identical setup to notebook 03)

In [ ]:
DATA_DIR = Path('../data/raw/ml-latest-small')

ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
movies  = pd.read_csv(DATA_DIR / 'movies.csv')

ALL_GENRES = sorted(
    g for g in movies.genres.str.split('|').explode().unique()
    if g != '(no genres listed)'
)

for g in ALL_GENRES:
    movies[g] = movies.genres.str.contains(g, regex=False).astype(int)

ratings = ratings.merge(movies[['movieId'] + ALL_GENRES], on='movieId', how='left')

user_mapping = build_id_mappings(ratings, 'userId')
item_mapping = build_id_mappings(ratings, 'movieId')
n_users, n_items = len(user_mapping), len(item_mapping)
field_dims = [n_users, n_items] + [2] * len(ALL_GENRES)

train_df, test_df = train_test_split_by_time(ratings)
print(f'n_users={n_users} | n_items={n_items} | train={len(train_df):,} | test={len(test_df):,}')

## 2. Dataset + Loaders

In [ ]:
class MovieLensDataset(Dataset):
    def __init__(self, df, user_mapping, item_mapping, genre_cols):
        self.users   = df['userId'].map(user_mapping).values
        self.items   = df['movieId'].map(item_mapping).values
        self.genres  = df[genre_cols].values.astype(np.int64)
        self.ratings = df['rating'].values.astype(np.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        x = np.concatenate([[self.users[idx], self.items[idx]], self.genres[idx]])
        return torch.tensor(x, dtype=torch.long), torch.tensor(self.ratings[idx])

BATCH = 512
train_ds = MovieLensDataset(train_df, user_mapping, item_mapping, ALL_GENRES)
test_ds  = MovieLensDataset(test_df,  user_mapping, item_mapping, ALL_GENRES)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH)

## 3. Shared Training Loop

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += criterion(pred, y).item() * len(y)
            total_n += len(y)
    return (total_loss / total_n) ** 0.5


def train_model(model, train_loader, test_loader, label='model', epochs=20, lr=1e-3, device=DEVICE):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history = {'train_rmse': [], 'test_rmse': []}
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            total_n += len(y)
        scheduler.step()
        train_rmse = (total_loss / total_n) ** 0.5
        test_rmse  = evaluate(model, test_loader, criterion, device)
        history['train_rmse'].append(train_rmse)
        history['test_rmse'].append(test_rmse)
        if epoch % 5 == 0 or epoch == 1:
            print(f'[{label}] Epoch {epoch:02d}/{epochs} | train {train_rmse:.4f} | test {test_rmse:.4f}')
    return model, history

## 4. Train FM (re-run for fair comparison with same seed)

In [ ]:
torch.manual_seed(SEED)
fm_model = FactorizationMachine(field_dims=field_dims, embed_dim=16)
print(f'FM params: {sum(p.numel() for p in fm_model.parameters()):,}')
fm_model, fm_history = train_model(fm_model, train_loader, test_loader, label='FM', epochs=20)

## 5. Train DeepFM

In [ ]:
torch.manual_seed(SEED)
deepfm_model = DeepFM(
    field_dims=field_dims,
    embed_dim=16,
    mlp_dims=[128, 64, 32],
    dropout=0.2
)
print(f'DeepFM params: {sum(p.numel() for p in deepfm_model.parameters()):,}')
deepfm_model, deepfm_history = train_model(deepfm_model, train_loader, test_loader, label='DeepFM', epochs=20)

## 6. Comparison — Loss Curves + Table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, split in zip(axes, ['train_rmse', 'test_rmse']):
    ax.plot(fm_history[split],     label='FM')
    ax.plot(deepfm_history[split], label='DeepFM')
    ax.set_title(split.replace('_', ' ').title())
    ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.legend()

plt.suptitle('FM vs DeepFM — Training Curves', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fm_best     = min(fm_history['test_rmse'])
deepfm_best = min(deepfm_history['test_rmse'])
improvement = (fm_best - deepfm_best) / fm_best * 100

comparison = pd.DataFrame({
    'Model'           : ['FM', 'DeepFM'],
    'Best Test RMSE'  : [round(fm_best, 4), round(deepfm_best, 4)],
    'Parameters'      : [
        sum(p.numel() for p in fm_model.parameters()),
        sum(p.numel() for p in deepfm_model.parameters())
    ],
    'Architecture'    : ['Linear + 2nd-order FM interactions', 'FM + MLP[128,64,32] on shared embeddings'],
})
display(comparison)
print(f'DeepFM improvement over FM: {improvement:.2f}%')

## 7. Save All Artifacts

In [ ]:
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

# --- FM ---
torch.save(fm_model.state_dict(), MODELS_DIR / 'fm_model.pt')
fm_cfg = {
    'model_type'   : 'fm',
    'field_dims'   : field_dims,
    'embed_dim'    : 16,
    'genre_cols'   : ALL_GENRES,
    'user_mapping' : {str(k): v for k, v in user_mapping.items()},
    'item_mapping' : {str(k): v for k, v in item_mapping.items()},
}
with open(MODELS_DIR / 'fm_config.json', 'w') as f:
    json.dump(fm_cfg, f, indent=2)

# --- DeepFM ---
torch.save(deepfm_model.state_dict(), MODELS_DIR / 'deepfm_model.pt')
deepfm_cfg = {
    'model_type'   : 'deepfm',
    'field_dims'   : field_dims,
    'embed_dim'    : 16,
    'mlp_dims'     : [128, 64, 32],
    'dropout'      : 0.2,
    'genre_cols'   : ALL_GENRES,
    'user_mapping' : {str(k): v for k, v in user_mapping.items()},
    'item_mapping' : {str(k): v for k, v in item_mapping.items()},
}
with open(MODELS_DIR / 'deepfm_config.json', 'w') as f:
    json.dump(deepfm_cfg, f, indent=2)

# --- Item embeddings from winning model ---
winner = deepfm_model if deepfm_best <= fm_best else fm_model
winner_name = 'DeepFM' if deepfm_best <= fm_best else 'FM'
print(f'Winner: {winner_name} — saving its item embeddings')
item_embeddings = winner.get_item_embeddings(
    item_field_index=1,
    item_offset_within_field=range(n_items)
).numpy()
np.save(MODELS_DIR / 'item_embeddings.npy', item_embeddings)

# --- Comparison CSV ---
comparison.to_csv(MODELS_DIR / 'comparison.csv', index=False)

print('\nSaved artifacts:')
for f in sorted(MODELS_DIR.iterdir()):
    if not f.name.startswith('.'):
        print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

## 8. Update backend/config.yaml to serve the winning model

Change the following lines in `backend/config.yaml` to point at the winning artifacts:

```yaml
model:
  fm_model_path:  "models/deepfm_model.pt"   # or fm_model.pt if FM wins
  fm_config_path: "models/deepfm_config.json" # or fm_config.json
```

The backend's `model_type` key in the config JSON tells `fm_inference.py` which class to instantiate — no other code change needed.